# 2-5. 데이터 품질과 모델 지표 연결 실습

이 notebook은 2-4에서 관측한 validation metric 변화를 데이터 품질 신호와 연결하는 실습입니다. 수강생은 “모델이 나빠졌다”라고 단정하는 사람이 아니라, 같은 모델과 같은 threshold에서 데이터 조건 변화가 score, prediction, FP/FN 변화와 함께 나타나는지 확인해야 하는 QA 담당자입니다.

핵심 흐름은 `validation data brief → GE 검증 실패 → feature 품질 비교 → prepared model metric artifact → 원인 후보 → QA report sentence`입니다. JupyterLite에서는 Great Expectations runtime이나 sklearn 모델을 직접 실행하지 않고, 준비된 artifact를 읽어 QA 판단에 사용합니다. 일반 notebook에서도 이 구조는 동일하게 유효하며, full runtime 재생성은 별도 GE/MLflow 일반 notebook 또는 로컬 Demo companion에서 다룹니다.

| 받은 업무 | 현장 관측값 | 결정 압박 |
| --- | --- | --- |
| validation 지표 저하가 데이터 조건 변화로 설명 가능한지 확인합니다 | 2-4에서 품질 저하 validation의 FP/FN 증가와 metric 하락이 관측되었습니다 | 모델 변경, 데이터 재확인, threshold 검토 중 무엇을 다음 action으로 남길지 정해야 합니다 |

| 이번 장에서 만드는 증거 | 보고서에 남길 산출물 | 다음 장으로 넘길 질문 |
| --- | --- | --- |
| GE failure table, data quality delta, score/prediction distribution, metric delta, cause candidate table | `chapter_02_data_metric_packet`, 원인 후보 강도와 owner | serving API가 같은 model_version, threshold, feature contract를 쓰고 있는가 |


### 따라하기 안내

이 notebook은 셀을 위에서 아래로 실행하면서 결과를 해석하는 실습입니다. 코드를 모두 이해하는 것보다, 각 셀이 만드는 근거를 보고 QA 판단으로 바꾸는 것이 중요합니다.

**오늘의 목표:** 데이터 품질 신호와 모델 metric 변화가 어떻게 연결되는지 확인합니다.

진행 방법은 단순합니다.

1. 안내 문장을 읽고 이 셀이 무엇을 확인하는지 파악합니다.
2. 바로 아래 코드 셀을 실행합니다.
3. 출력에서 핵심 값 1-2개만 고릅니다.
4. 그 값을 보고서에 쓸 수 있는 문장으로 바꿉니다.

마지막에는 다음 형태로 정리합니다.

```text
데이터 조건 변화와 metric 변화가 함께 관측되며, 원인 확정이 아니라 다음 확인 후보로 남깁니다.
```


## 1. 실습 준비와 artifact 범위 고정

### 1-1. 일반/Lite 실행 기준과 prepared artifact 역할 확인

이 notebook의 첫 판단은 어떤 결과를 직접 계산하고 어떤 결과를 prepared artifact로 확인하는지 분리하는 것입니다. CSV 데이터 품질 요약은 notebook에서 직접 계산하지만, Great Expectations 검증 결과와 sklearn 기반 모델 metric은 준비된 JSON/Markdown artifact를 읽습니다.

이 분리는 브라우저 제약 때문만이 아닙니다. QA 보고서에서는 “직접 실행한 값”과 “준비된 artifact에서 확인한 값”을 구분해야 감사 가능성이 생깁니다.

준비 셀은 package import와 경로 helper만 담당합니다. 데이터 품질 계산, artifact provenance, 원인 후보 연결은 뒤 셀에서 Pandas 표로 드러냅니다.


### 따라하기 안내: 비교 범위 고정

**이 셀에서 할 일**  
필요한 경로와 helper를 준비합니다.

**실행 후 볼 것**  
artifact 경로와 실행 범위를 확인합니다.

**기록 포인트**  
prepared artifact인지 local 재생성인지 적습니다.

**생각해 볼 질문**  
지금 보는 비교는 어떤 데이터셋 사이의 비교인가요?


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import utils as ch02

prepared = await ch02.prepare_notebook()
pd = prepared.pandas
aiq_lite = prepared.aiq_lite

FEATURE_COLUMNS = aiq_lite.FEATURE_COLUMNS
POSITIVE_LABEL = aiq_lite.POSITIVE_LABEL
NEGATIVE_LABEL = aiq_lite.NEGATIVE_LABEL
THRESHOLD = aiq_lite.THRESHOLD


def resolve_existing_path(path: str) -> Path | None:
    for candidate in [Path(path), Path("..") / path, Path("../..") / path]:
        if candidate.exists():
            return candidate
    return None


def load_json_artifact(path: str) -> tuple[dict[str, object], str]:
    candidate = resolve_existing_path(path)
    if candidate is None:
        return {}, f"missing {path}"
    return json.loads(candidate.read_text(encoding="utf-8")), str(candidate)


def load_text_artifact(path: str) -> tuple[str, str]:
    candidate = resolve_existing_path(path)
    if candidate is None:
        return "", f"missing {path}"
    return candidate.read_text(encoding="utf-8"), str(candidate)

environment_summary = pd.DataFrame(
    [
        {"item": "package_module", "value": aiq_lite.__name__, "why_it_matters": "Lite와 로컬에서 같은 label/metric 기준을 제공합니다."},
        {"item": "positive_label", "value": POSITIVE_LABEL, "why_it_matters": "recall과 precision을 계산할 관심 class입니다."},
        {"item": "negative_label", "value": NEGATIVE_LABEL, "why_it_matters": "FP를 해석할 비교 class입니다."},
        {"item": "operating_threshold", "value": THRESHOLD, "why_it_matters": "score를 prediction으로 바꾸는 기준입니다."},
    ]
)

display(environment_summary)


이 출력에서 확인할 핵심은 실행 환경보다 evidence 경계입니다. `ai_quality.lite`는 CSV 로딩과 공통 label/feature 기준을 제공하지만, GE 결과와 sklearn metric은 artifact에서 읽습니다.

따라서 이 notebook의 보고 문장에는 두 종류의 근거가 섞입니다. 데이터 품질 delta는 현재 notebook 실행값이고, full model metric delta는 prepared artifact 확인값입니다.

## 2. validation 데이터 brief와 label 기준

### 2-1. baseline/degraded validation 원본 데이터 고정

이 셀의 판단은 어떤 두 데이터셋을 비교하는지 metric 전에 고정하는 것입니다. `valid_baseline`과 `valid_degraded`는 같은 row grain을 가진 validation classification sample이며, degraded 데이터는 운영 입력 실패 양상을 교육용으로 재현한 비교 artifact입니다.

한 row는 생체신호 기반 위험 알림 시스템의 평가용 sample 하나를 뜻합니다. 이 데이터는 실제 의료 판단이 아니라 AI QA classification 예제입니다.

이 단계에서는 score와 metric을 계산하지 않습니다. 먼저 source, sample scope, row count, raw label 분포, preview를 확인해 비교 대상이 같은 의미인지 판단합니다.


### 따라하기 안내: validation 데이터 목록

**이 셀에서 할 일**  
baseline과 degraded validation을 명시합니다.

**실행 후 볼 것**  
비교 대상 이름을 확인합니다.

**기록 포인트**  
모델과 threshold를 고정한 비교라고 적습니다.

**생각해 볼 질문**  
비교 조건이 고정되지 않으면 무엇이 어려워질까요?


In [ ]:
VALIDATION_DATASETS = [
    ("valid_baseline", "data/vital_signs_valid_baseline.csv"),
    ("valid_degraded", "data/vital_signs_valid_degraded.csv"),
]

raw_validation_frames: dict[str, pd.DataFrame] = {}
provenance_rows: list[dict[str, object]] = []
preview_frames: list[pd.DataFrame] = []
raw_label_frames: list[pd.DataFrame] = []

for dataset_name, dataset_path in VALIDATION_DATASETS:
    raw_frame, data_source = aiq_lite.load_csv_or_sample(
        dataset_path,
        aiq_lite.sample_vital_signs(),
        nrows=None,
    )
    raw_validation_frames[dataset_name] = raw_frame
    execution_scope = "embedded_fallback_sample" if "embedded" in data_source else (
        "jupyterlite_sample" if len(raw_frame) <= 600 else "local_full_or_large_sample"
    )
    provenance_rows.append(
        {
            "dataset": dataset_name,
            "dataset_role": "validation_comparison_input",
            "path": dataset_path,
            "data_source": data_source,
            "execution_scope": execution_scope,
            "row_grain": "one validation classification sample",
            "row_count": raw_frame.shape[0],
            "column_count": raw_frame.shape[1],
        }
    )
    preview_columns = [column for column in ["patient_id", "timestamp", *FEATURE_COLUMNS, "label"] if column in raw_frame.columns]
    preview_frames.append(
        raw_frame.loc[:, preview_columns]
        .head(3)
        .assign(dataset=dataset_name)
        .loc[:, ["dataset", *preview_columns]]
    )
    raw_label_frames.append(
        raw_frame["label"]
        .value_counts(dropna=False)
        .rename_axis("raw_label")
        .reset_index(name="count")
        .assign(
            dataset=dataset_name,
            ratio_pct=lambda table, denominator=len(raw_frame): (table["count"] / denominator * 100).round(2),
        )
    )

validation_provenance = pd.DataFrame(provenance_rows)
validation_raw_preview = pd.concat(preview_frames, ignore_index=True)
validation_raw_label_distribution = pd.concat(raw_label_frames, ignore_index=True).loc[
    :, ["dataset", "raw_label", "count", "ratio_pct"]
]

comparison_scope = pd.DataFrame(
    [
        {"check": "same_row_grain", "observed": "one validation classification sample", "qa_use": "metric delta의 분모가 같은 의미인지 확인합니다."},
        {"check": "comparison_scope", "observed": "validation artifact, not live operation log", "qa_use": "운영 원인 확정이 아니라 후보 강화 근거로 사용합니다."},
        {"check": "score_status", "observed": "not_created_in_this_cell", "qa_use": "metric 전에 데이터 조건 차이를 먼저 확인합니다."},
    ]
)


### 출력 확인: `validation_provenance`

**읽는 순서**  
먼저 `source` 또는 `path`를 보고, 그 다음 `execution_scope`나 데이터셋 이름을 확인합니다. 이 출력은 숫자를 해석하기 전에 “어떤 근거를 보고 있는지”를 고정하는 단계입니다.

**해석 기준**  
경로가 prepared artifact이면 준비된 근거를 읽은 것이고, 로컬 파일이면 현재 환경에서 읽은 근거입니다. 실행 범위가 sample이면 뒤의 metric 숫자를 전체 데이터 결과처럼 쓰면 안 됩니다.

**보고서 문장 예시**  
“이 결과는 `{path}` 경로의 prepared/local artifact를 기준으로 확인했으며, 실행 범위는 출력의 scope를 따릅니다.”

**주의할 점**  
값이 좋아 보이거나 나빠 보여도, 출처와 범위를 먼저 적지 않으면 보고서에서 재현 가능한 근거가 되기 어렵습니다.


In [ ]:
display(validation_provenance)


### 출력 확인: `comparison_scope`

**읽는 순서**  
행 수, 컬럼 수, 데이터셋 이름, 실행 범위를 순서대로 봅니다. 여기서는 아직 품질 판단을 확정하지 않고 “분석 대상이 맞는지”를 확인합니다.

**해석 기준**  
행 수가 예상보다 작으면 JupyterLite sample 또는 fallback sample일 수 있습니다. 컬럼 수나 데이터셋 이름이 기대와 다르면 뒤의 결측, label, metric 해석이 모두 흔들립니다.

**보고서 문장 예시**  
“분석 대상은 출력에 표시된 데이터셋이며, row/column 수와 실행 범위를 확인한 뒤 다음 변환으로 진행했습니다.”

**주의할 점**  
이 표는 성능이나 품질 결론이 아닙니다. 데이터 출발점을 고정하는 근거입니다.


In [ ]:
display(comparison_scope)


### 출력 확인: `validation_raw_preview`

**읽는 순서**  
왼쪽부터 컬럼 이름을 보고, 한 행이 무엇을 의미하는지 확인합니다. 그 다음 feature, label, score, prediction 같은 주요 컬럼이 기대한 위치에 있는지 봅니다.

**해석 기준**  
Preview는 전체 통계가 아니라 샘플 행입니다. 값의 모양, 단위, 컬럼 이름, 변환 후 새 컬럼이 생겼는지를 확인하는 용도입니다.

**보고서 문장 예시**  
“Preview에서 주요 feature와 label/prediction 컬럼이 기대한 구조로 존재함을 확인했습니다.”

**주의할 점**  
앞 5개 행만 보고 분포나 성능을 판단하면 안 됩니다. 분포와 metric은 뒤의 별도 출력에서 확인합니다.


In [ ]:
display(validation_raw_preview)


### 출력 확인: `validation_raw_label_distribution`

**읽는 순서**  
`high_risk`와 `low_risk`의 count와 ratio를 봅니다. `high_risk`는 positive class이므로 표본 수가 recall 해석의 기본 조건입니다.

**해석 기준**  
두 class가 모두 존재하면 metric 계산은 가능합니다. 한 class가 너무 적으면 accuracy보다 precision/recall과 support를 함께 읽어야 합니다.

**보고서 문장 예시**  
“표준화된 label 분포에서 positive/negative class가 모두 확인되어 metric 계산 기준을 세울 수 있습니다.”

**주의할 점**  
class 비율이 균형적이라고 해서 데이터 품질이 모두 좋다는 뜻은 아닙니다. 결측, 범위, label validity는 별도로 봅니다.


In [ ]:
display(validation_raw_label_distribution)


이 출력에서 확인할 핵심은 비교 대상이 같은 의미의 validation sample이라는 점입니다. row grain이 다르면 metric 변화가 데이터 조건 변화인지 비교 대상 차이인지 분리할 수 없습니다.

`execution_scope`가 sample이면 보고서에는 sample 기준이라고 적어야 합니다. 전체 로컬 데이터와 Lite sample은 같은 판단 흐름을 연습할 수 있지만, 숫자 규모는 다를 수 있습니다.

### 2-2. label 표준화와 class support 비교

이 셀의 판단은 두 validation 데이터셋의 label이 같은 계약으로 해석되는지 확인하는 것입니다. label은 정답 기준이므로, feature 품질이나 score를 보기 전에 raw label이 `high_risk`, `low_risk`로 표준화되는 과정을 분리해서 봅니다.

라벨 허용값 검증이 통과했다는 말은 label 값이 허용 목록 안에 있다는 뜻입니다. 모든 행의 정답 기준이 실제 상태와 완벽하게 일치한다는 뜻은 아니므로, label flip 후보는 별도 제한 사항으로 남깁니다.

이 단계의 통과 기준은 허용되지 않은 label이 없고, 두 class support가 모두 존재하는 것입니다. 통과하더라도 degraded 데이터의 class 비율 변화는 metric 해석의 제한 사항입니다.


### 따라하기 안내: validation 데이터 로딩

**이 셀에서 할 일**  
두 데이터셋을 같은 방식으로 읽습니다.

**실행 후 볼 것**  
row 수와 주요 label 분포를 봅니다.

**기록 포인트**  
같은 로딩 기준으로 비교했다고 적습니다.

**생각해 볼 질문**  
데이터 로딩 방식이 다르면 metric 차이를 어떻게 해석해야 할까요?


In [ ]:
validation_frames: dict[str, pd.DataFrame] = {}
label_transition_frames: list[pd.DataFrame] = []
standard_label_frames: list[pd.DataFrame] = []
label_gate_rows: list[dict[str, object]] = []

for dataset_name, raw_frame in raw_validation_frames.items():
    standardized_frame = raw_frame.copy().assign(
        label=lambda table: table["label"].map(aiq_lite.normalize_label),
        dataset=dataset_name,
    )
    validation_frames[dataset_name] = standardized_frame
    label_transition_frames.append(
        raw_frame.loc[:, ["label"]]
        .rename(columns={"label": "raw_label"})
        .assign(standardized_label=standardized_frame["label"], dataset=dataset_name)
        .value_counts(dropna=False)
        .rename("count")
        .reset_index()
    )
    distribution = (
        standardized_frame["label"]
        .value_counts(dropna=False)
        .rename_axis("standardized_label")
        .reset_index(name="count")
        .assign(
            dataset=dataset_name,
            ratio_pct=lambda table, denominator=len(standardized_frame): (table["count"] / denominator * 100).round(2),
        )
    )
    standard_label_frames.append(distribution)
    class_support = distribution.set_index("standardized_label")["count"].to_dict()
    unexpected_label_count = int(~standardized_frame["label"].isin([POSITIVE_LABEL, NEGATIVE_LABEL]).sum())
    label_gate_rows.append(
        {
            "dataset": dataset_name,
            "unexpected_label_count": unexpected_label_count,
            "positive_support": int(class_support.get(POSITIVE_LABEL, 0)),
            "negative_support": int(class_support.get(NEGATIVE_LABEL, 0)),
            "status": "pass" if unexpected_label_count == 0 and all(class_support.get(label, 0) > 0 for label in [POSITIVE_LABEL, NEGATIVE_LABEL]) else "hold",
            "qa_judgment": "허용 label 기준으로 metric 계산 형식은 유지됩니다."
            if unexpected_label_count == 0 else "metric 계산 전에 label 계약을 확인해야 합니다.",
        }
    )

label_transition = pd.concat(label_transition_frames, ignore_index=True).loc[
    :, ["dataset", "raw_label", "standardized_label", "count"]
]
standardized_label_distribution = pd.concat(standard_label_frames, ignore_index=True).loc[
    :, ["dataset", "standardized_label", "count", "ratio_pct"]
]
label_gate = pd.DataFrame(label_gate_rows)
label_limit_note = pd.DataFrame(
    [
        {
            "limit": "allowed_label_check_is_not_ground_truth_audit",
            "meaning": "허용 라벨 검증은 값의 형식만 확인합니다.",
            "qa_action": "label flip 후보는 metric 해석 제한 사항으로 별도 기록합니다.",
        }
    ]
)


### 출력 확인: `label_transition`

**읽는 순서**  
원본 label 값과 표준화된 label 값을 나란히 봅니다. 같은 의미의 값이 `high_risk`, `low_risk`로 정리되는지 확인합니다.

**해석 기준**  
표준화 후 label이 비거나 예상 밖 값이 생기면 metric 계산 전에 멈춰야 합니다. label은 모델 평가의 정답 기준이므로 작은 표기 차이도 FP/FN 해석을 바꿀 수 있습니다.

**보고서 문장 예시**  
“원본 label은 평가용 label 기준으로 표준화되었고, 변환 전후 값을 추적할 수 있습니다.”

**주의할 점**  
표준화 결과만 보지 말고 원본 값이 무엇이었는지도 같이 확인해야 나중에 label mapping 문제를 추적할 수 있습니다.


In [ ]:
display(label_transition)


### 출력 확인: `standardized_label_distribution`

**읽는 순서**  
`high_risk`와 `low_risk`의 count와 ratio를 봅니다. `high_risk`는 positive class이므로 표본 수가 recall 해석의 기본 조건입니다.

**해석 기준**  
두 class가 모두 존재하면 metric 계산은 가능합니다. 한 class가 너무 적으면 accuracy보다 precision/recall과 support를 함께 읽어야 합니다.

**보고서 문장 예시**  
“표준화된 label 분포에서 positive/negative class가 모두 확인되어 metric 계산 기준을 세울 수 있습니다.”

**주의할 점**  
class 비율이 균형적이라고 해서 데이터 품질이 모두 좋다는 뜻은 아닙니다. 결측, 범위, label validity는 별도로 봅니다.


In [ ]:
display(standardized_label_distribution)


### 출력 확인: `label_gate`

**읽는 순서**  
invalid label, missing label, positive support, negative support를 차례로 봅니다.

**해석 기준**  
invalid 또는 missing label이 있으면 정답 비교가 흔들립니다. positive support가 부족하면 recall 한두 건 차이도 큰 비율 변화로 보일 수 있습니다.

**보고서 문장 예시**  
“label gate에서 허용 label과 class support를 확인했으며, metric 해석 가능 범위를 판단했습니다.”

**주의할 점**  
label gate 통과는 feature 품질 통과를 의미하지 않습니다. label은 정답 기준만 확인합니다.


In [ ]:
display(label_gate)


### 출력 확인: `label_limit_note`

**읽는 순서**  
note의 문장을 읽고, label 기준에서 어떤 제한 사항이 남는지 확인합니다.

**해석 기준**  
이 출력은 숫자보다 해석 문장이 중요합니다. label 기준이 불완전하면 뒤의 metric도 제한적으로 해석해야 합니다.

**보고서 문장 예시**  
“label 기준의 제한 사항은 note에 따라 보고서 caveat로 남깁니다.”

**주의할 점**  
note가 비어 있거나 pass처럼 보여도, 데이터셋 범위가 sample인지 full인지 같이 확인합니다.


In [ ]:
display(label_limit_note)


이 출력에서 확인할 핵심은 label 계약은 metric 계산 형식을 지탱하지만, 정답 기준 신뢰 전체를 보장하지 않는다는 점입니다. 따라서 2-5의 결론은 “label 문제가 없다”가 아니라 “허용 label 기준은 유지되지만 label basis review 후보는 남는다”가 되어야 합니다.

이제 feature 품질 신호를 확인합니다. feature 품질은 score와 prediction 분포를 흔들 수 있으므로, metric delta를 해석하기 전에 먼저 봐야 합니다.

## 3. Great Expectations artifact와 feature 품질 신호

### 3-1. GE 검증 결과 artifact 읽기

이 셀의 판단은 prepared GE artifact가 어떤 데이터 기준 실패를 남겼는지 확인하는 것입니다. JupyterLite에서는 Great Expectations runtime을 실행하지 않고, 이미 생성된 expectation/result/summary artifact를 읽습니다.

artifact 확인은 도구 사용법 실습이 아니라 auditability 확인입니다. 어떤 기준이 실패했고, 실패 건수와 비율이 얼마이며, 그 실패가 label 기준인지 feature 기준인지 분리해야 합니다.

이 결과는 metric 변화의 확정 원인이 아닙니다. 하지만 feature 결측이나 범위 오류가 보이면 score와 prediction 변화의 원인 후보로 강화됩니다.


### 따라하기 안내: GE 검증 결과

**이 셀에서 할 일**  
품질 rule 실패 결과를 읽습니다.

**실행 후 볼 것**  
실패 column과 rule을 확인합니다.

**기록 포인트**  
데이터 품질 이슈 후보를 적습니다.

**생각해 볼 질문**  
실패한 rule은 metric 변화의 어떤 후보가 될까요?


In [ ]:
GE_EXPECTATIONS_PATH = "artifacts/great_expectations/chapter_02_expectations.json"
GE_RESULT_PATH = "artifacts/great_expectations/chapter_02_validation_result.json"
GE_SUMMARY_PATH = "artifacts/great_expectations/chapter_02_validation_summary.md"

expectations_json, expectations_source = load_json_artifact(GE_EXPECTATIONS_PATH)
ge_result_json, ge_result_source = load_json_artifact(GE_RESULT_PATH)
ge_summary_text, ge_summary_source = load_text_artifact(GE_SUMMARY_PATH)

expectation_table = pd.DataFrame(expectations_json.get("expectations", []))
ge_result_table = pd.DataFrame(ge_result_json.get("expectation_results", []))
if not ge_result_table.empty:
    ge_result_table = ge_result_table.assign(
        status=lambda table: table["success"].map({True: "pass", False: "fail"}),
        unexpected_ratio_pct=lambda table: table["unexpected_ratio"].round(2),
        evidence_role=lambda table: table["column"].map(lambda column: "label_contract" if column == "label" else "feature_quality"),
    )

ge_artifact_provenance = pd.DataFrame(
    [
        {"artifact": "expectations", "path": expectations_source, "qa_use": "검증 기준 목록을 확인합니다."},
        {"artifact": "validation_result", "path": ge_result_source, "qa_use": "실패 여부, 실패 건수, 실패 비율을 확인합니다."},
        {"artifact": "validation_summary", "path": ge_summary_source, "qa_use": "보고서에 인용할 요약을 확인합니다."},
    ]
)

ge_overall_summary = pd.DataFrame(
    [
        {
            "dataset_name": ge_result_json.get("dataset_name", "not_available"),
            "row_count": ge_result_json.get("row_count", 0),
            "overall_success": ge_result_json.get("success", None),
            "success_count": ge_result_json.get("success_count", 0),
            "failure_count": ge_result_json.get("failure_count", 0),
        }
    ]
)

ge_failure_table = ge_result_table.loc[
    ge_result_table["success"].eq(False),
    ["expectation_type", "column", "status", "unexpected_count", "unexpected_ratio_pct", "observed_value", "qa_reason", "evidence_role"],
] if not ge_result_table.empty else pd.DataFrame()


### 출력 확인: `ge_artifact_provenance`

**읽는 순서**  
먼저 `source` 또는 `path`를 보고, 그 다음 `execution_scope`나 데이터셋 이름을 확인합니다. 이 출력은 숫자를 해석하기 전에 “어떤 근거를 보고 있는지”를 고정하는 단계입니다.

**해석 기준**  
경로가 prepared artifact이면 준비된 근거를 읽은 것이고, 로컬 파일이면 현재 환경에서 읽은 근거입니다. 실행 범위가 sample이면 뒤의 metric 숫자를 전체 데이터 결과처럼 쓰면 안 됩니다.

**보고서 문장 예시**  
“이 결과는 `{path}` 경로의 prepared/local artifact를 기준으로 확인했으며, 실행 범위는 출력의 scope를 따릅니다.”

**주의할 점**  
값이 좋아 보이거나 나빠 보여도, 출처와 범위를 먼저 적지 않으면 보고서에서 재현 가능한 근거가 되기 어렵습니다.


In [ ]:
display(ge_artifact_provenance)


### 출력 확인: `ge_overall_summary`

**읽는 순서**  
status, pass/fail, note, failed count를 순서대로 봅니다. 어떤 기준이 통과했고 어떤 기준이 제한 사항인지 나눕니다.

**해석 기준**  
gate 결과는 최종 결론이 아니라 다음 단계로 넘어갈 수 있는지 판단하는 checkpoint입니다. fail이나 warning은 보고서 caveat와 next action으로 연결합니다.

**보고서 문장 예시**  
“품질 gate 요약에서 통과 항목과 제한 항목을 구분했고, 이후 metric 해석 범위에 반영했습니다.”

**주의할 점**  
pass만 보고 끝내지 않습니다. 어떤 기준을 통과한 것인지, 아직 확인하지 않은 영역은 무엇인지 남겨야 합니다.


In [ ]:
display(ge_overall_summary)


### 출력 확인: `ge_result_table.loc[:, ["expectation_type", "column", "status", "unex...`

**읽는 순서**  
expectation type, column, status, unexpected count를 봅니다. 실패한 rule이 어떤 column에 걸렸는지 먼저 확인합니다.

**해석 기준**  
GE 결과는 도구 화면이 아니라 품질 기준의 pass/fail 기록입니다. 실패한 column은 feature 품질 또는 label 기준 문제의 후보가 됩니다.

**보고서 문장 예시**  
“검증 rule별 결과에서 실패 column과 실패 기준을 확인했고, 데이터 품질 조치 후보로 기록했습니다.”

**주의할 점**  
GE 실패가 곧 모델 성능 저하의 확정 원인은 아닙니다. metric 변화와 연결해 후보로 해석합니다.


In [ ]:
display(ge_result_table.loc[:, ["expectation_type", "column", "status", "unexpected_count", "unexpected_ratio_pct", "observed_value", "qa_reason", "evidence_role"]])


### 출력 확인: `ge_failure_table`

**읽는 순서**  
실패한 rule만 모아 column, expectation, 실패 규모를 확인합니다.

**해석 기준**  
이 표는 조치 우선순위를 잡는 데 씁니다. 어떤 column이 반복적으로 실패하면 데이터 수집 또는 전처리 owner에게 확인이 필요합니다.

**보고서 문장 예시**  
“실패 rule 목록에서 조치가 필요한 column을 확인했고, metric 변화의 후보 근거로 남겼습니다.”

**주의할 점**  
실패 목록만 보고 원인을 확정하지 않습니다. 데이터 생성 경로와 metric delta를 함께 봅니다.


In [ ]:
display(ge_failure_table)


이 출력에서 확인할 핵심은 `heart_rate` 결측과 `oxygen_saturation` 범위 오류가 feature 품질 실패로 분류된다는 점입니다. 반대로 `label` 결측과 허용 label 기준은 통과했으므로, label 형식 때문에 metric 계산을 즉시 보류할 근거는 약합니다.

다만 label 형식 통과는 label basis 전체 신뢰를 뜻하지 않습니다. 따라서 보고서에는 “정답 기준 형식은 유지되지만, 교육용 degraded 데이터의 label flip 후보는 제한 사항으로 남긴다”라고 적어야 합니다.

### 3-2. validation CSV에서 feature 품질 delta 직접 계산

이 셀의 판단은 GE failure를 notebook의 dataframe evidence로 재확인하는 것입니다. artifact만 읽으면 수강생은 도구 결과를 그대로 믿게 되므로, Pandas로 결측, 범위 오류, 핵심 feature 분포를 다시 계산합니다.

직접 계산은 GE를 대체하려는 것이 아니라 artifact 해석을 학습 가능한 evidence로 바꾸는 과정입니다. `missing_count`, `invalid_count`, feature median/min/max를 보면 어떤 feature가 score와 prediction 분포를 흔들 수 있는지 설명할 수 있습니다.

여기서 중요한 것은 degraded 데이터가 baseline보다 어떤 방향으로 달라졌는지입니다. 이 차이가 뒤의 score/prediction/metric delta와 같은 방향인지 확인해야 원인 후보가 강해집니다.


### 따라하기 안내: feature 품질 신호

**이 셀에서 할 일**  
결측, 범위 오류 같은 feature 품질 신호를 정리합니다.

**실행 후 볼 것**  
문제가 있는 feature와 개수를 봅니다.

**기록 포인트**  
score에 영향을 줄 수 있는 feature를 적습니다.

**생각해 볼 질문**  
어떤 feature 문제가 score 분포를 움직일 수 있나요?


In [ ]:
feature_quality_rows: list[dict[str, object]] = []
range_quality_rows: list[dict[str, object]] = []
feature_distribution_rows: list[dict[str, object]] = []

for dataset_name, frame in validation_frames.items():
    numeric_features = frame.loc[:, FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce")
    rows_with_any_feature_missing = int(frame.loc[:, FEATURE_COLUMNS].isna().any(axis=1).sum())
    feature_quality_rows.append(
        {
            "dataset": dataset_name,
            "row_count": len(frame),
            "rows_with_any_feature_missing": rows_with_any_feature_missing,
            "missing_row_pct": round(rows_with_any_feature_missing / len(frame) * 100, 2),
            "heart_rate_missing": int(frame["heart_rate"].isna().sum()),
            "oxygen_saturation_missing": int(frame["oxygen_saturation"].isna().sum()),
            "heart_rate_median": round(float(numeric_features["heart_rate"].median()), 4),
            "oxygen_saturation_min": round(float(numeric_features["oxygen_saturation"].min()), 4),
            "oxygen_saturation_max": round(float(numeric_features["oxygen_saturation"].max()), 4),
        }
    )
    for feature in ["heart_rate", "oxygen_saturation"]:
        values = numeric_features[feature]
        feature_distribution_rows.append(
            {
                "dataset": dataset_name,
                "feature": feature,
                "count": int(values.notna().sum()),
                "missing_count": int(values.isna().sum()),
                "mean": round(float(values.mean()), 4),
                "p10": round(float(values.quantile(0.10)), 4),
                "p50": round(float(values.quantile(0.50)), 4),
                "p90": round(float(values.quantile(0.90)), 4),
                "min": round(float(values.min()), 4),
                "max": round(float(values.max()), 4),
            }
        )
    for column, (minimum, maximum) in aiq_lite.VALID_RANGES.items():
        if column not in frame.columns:
            continue
        values = pd.to_numeric(frame[column], errors="coerce")
        invalid_count = int((values.notna() & ((values < minimum) | (values > maximum))).sum())
        range_quality_rows.append(
            {
                "dataset": dataset_name,
                "feature": column,
                "valid_min": minimum,
                "valid_max": maximum,
                "invalid_count": invalid_count,
                "invalid_pct": round(invalid_count / len(frame) * 100, 2),
            }
        )

feature_quality = pd.DataFrame(feature_quality_rows)
range_quality = pd.DataFrame(range_quality_rows).sort_values(
    ["invalid_count", "dataset", "feature"], ascending=[False, True, True]
)
feature_distribution = pd.DataFrame(feature_distribution_rows)

quality_delta_columns = [
    "rows_with_any_feature_missing",
    "missing_row_pct",
    "heart_rate_missing",
    "oxygen_saturation_min",
    "oxygen_saturation_max",
]
quality_by_dataset = feature_quality.set_index("dataset")
quality_delta = (
    (quality_by_dataset.loc["valid_degraded", quality_delta_columns] - quality_by_dataset.loc["valid_baseline", quality_delta_columns])
    .to_frame()
    .T
    .round(4)
    .assign(comparison="valid_degraded_minus_baseline")
    .loc[:, ["comparison", *quality_delta_columns]]
)

quality_gate = pd.DataFrame(
    [
        {
            "gate": "feature_quality_comparison",
            "status": "review" if int(quality_delta["rows_with_any_feature_missing"].iloc[0]) > 0 or int(range_quality["invalid_count"].max()) > 0 else "pass",
            "qa_judgment": "degraded 데이터의 feature 품질 차이가 score와 metric 변화의 원인 후보입니다."
            if int(quality_delta["rows_with_any_feature_missing"].iloc[0]) > 0 or int(range_quality["invalid_count"].max()) > 0
            else "두 데이터셋의 feature 품질 차이가 크지 않습니다.",
        }
    ]
)


### 출력 확인: `feature_quality`

**읽는 순서**  
feature별 missing, invalid, status를 행 단위로 봅니다. 문제가 있는 feature 이름을 먼저 고릅니다.

**해석 기준**  
문제가 있는 feature가 모델 score에 직접 들어가면 metric 변화의 원인 후보가 됩니다. 특히 vital sign feature의 결측과 범위 오류는 score 분포를 움직일 수 있습니다.

**보고서 문장 예시**  
“feature 품질 표에서 문제가 있는 입력 항목을 확인했고, metric 변화의 원인 후보로 남겼습니다.”

**주의할 점**  
feature 품질 문제를 바로 모델 결함으로 쓰지 않습니다. 먼저 데이터 수집, 전처리, 입력 계약 문제를 후보로 둡니다.


In [ ]:
display(feature_quality)


### 출력 확인: `quality_delta`

**읽는 순서**  
baseline 대비 증가/감소 방향을 봅니다. 양수와 음수가 각각 좋은 변화인지 나쁜 변화인지 항목별로 해석합니다.

**해석 기준**  
delta는 원인을 확정하지 않고 변화가 발생한 위치를 알려 줍니다. metric delta는 데이터 품질 delta, score delta와 함께 봐야 합니다.

**보고서 문장 예시**  
“baseline 대비 변화량에서 어떤 품질 신호와 metric이 함께 움직였는지 확인했습니다.”

**주의할 점**  
모든 delta의 부호가 같은 의미는 아닙니다. 예를 들어 recall delta 감소와 FP delta 증가는 서로 다른 오류 변화를 뜻합니다.


In [ ]:
display(quality_delta)


### 출력 확인: `feature_distribution`

**읽는 순서**  
feature별 평균, 최소, 최대 또는 분포 요약을 봅니다. 기준 데이터와 비교되는 경우 변화 방향을 확인합니다.

**해석 기준**  
평균이 움직이면 입력 구성 변화 후보가 됩니다. 하지만 평균만으로 분포 전체를 설명할 수 없으므로 histogram이나 range 결과와 같이 봐야 합니다.

**보고서 문장 예시**  
“feature 분포 요약에서 입력값의 중심과 범위를 확인했고, score 변화 가능성을 점검했습니다.”

**주의할 점**  
분포 요약은 원인 확정이 아니라 다음 확인 후보입니다. label과 metric 결과와 연결해서 해석합니다.


In [ ]:
display(feature_distribution)


### 출력 확인: `range_quality.loc[range_quality["invalid_count"].gt(0)].head(12)`

**읽는 순서**  
feature별 최소/최대와 invalid count를 봅니다. 도메인상 가능한 범위를 벗어난 값이 있는지 확인합니다.

**해석 기준**  
범위 오류는 API가 정상 응답해도 score를 비정상적으로 움직일 수 있습니다. 특히 산소포화도, 체온, 혈압 같은 feature는 값의 의미가 중요합니다.

**보고서 문장 예시**  
“range gate에서 허용 범위를 벗어난 feature를 확인했고, score/metric 해석의 제한 사항으로 기록했습니다.”

**주의할 점**  
이상값을 바로 제거한다고 결론 내리지 않습니다. 먼저 수집 오류인지, 단위 문제인지, 실제 rare case인지 확인해야 합니다.


In [ ]:
display(range_quality.loc[range_quality["invalid_count"].gt(0)].head(12))


### 출력 확인: `quality_gate`

**읽는 순서**  
status, pass/fail, note, failed count를 순서대로 봅니다. 어떤 기준이 통과했고 어떤 기준이 제한 사항인지 나눕니다.

**해석 기준**  
gate 결과는 최종 결론이 아니라 다음 단계로 넘어갈 수 있는지 판단하는 checkpoint입니다. fail이나 warning은 보고서 caveat와 next action으로 연결합니다.

**보고서 문장 예시**  
“품질 gate 요약에서 통과 항목과 제한 항목을 구분했고, 이후 metric 해석 범위에 반영했습니다.”

**주의할 점**  
pass만 보고 끝내지 않습니다. 어떤 기준을 통과한 것인지, 아직 확인하지 않은 영역은 무엇인지 남겨야 합니다.


In [ ]:
display(quality_gate)


이 출력에서 확인할 핵심은 데이터 품질 신호가 실제 dataframe에서도 재현된다는 점입니다. degraded 데이터에서 `heart_rate` 결측과 `oxygen_saturation` 범위 오류가 관측되면, score와 prediction 변화의 원인 후보가 강해집니다.

아직 모델 metric을 보지 않았으므로 결론은 보류합니다. 다음 단계에서는 prepared model artifact를 읽어 같은 모델과 같은 threshold에서 score/prediction/metric이 어떤 방향으로 움직였는지 확인합니다.

## 4. prepared model metric artifact와 분포 변화

### 4-1. 모델/threshold 조건과 score distribution 확인

이 셀의 판단은 metric 변화가 같은 모델과 같은 threshold 조건에서 계산되었는지 확인하는 것입니다. 모델 버전이나 threshold가 바뀌었다면 데이터 품질 신호와 metric 변화를 직접 연결하기 어렵습니다.

`validation_degradation_comparison.json`은 sklearn 기반 full evaluation artifact입니다. Lite notebook은 이 artifact를 재생성하지 않고 읽습니다. 이 제한을 명확히 해야 “브라우저에서 모델을 학습했다”는 잘못된 보고를 막을 수 있습니다.

먼저 model condition과 score/prediction distribution을 확인합니다. score 평균 하나가 아니라 p10, p50, p90, `high_risk` prediction count를 함께 봐야 metric 변화의 중간 경로를 설명할 수 있습니다.


### 따라하기 안내: 비교 artifact 읽기

**이 셀에서 할 일**  
모델 비교 JSON artifact를 읽습니다.

**실행 후 볼 것**  
metric, delta, distribution 항목을 확인합니다.

**기록 포인트**  
보고서 evidence path를 적습니다.

**생각해 볼 질문**  
이 artifact는 원인 확정인가요, 비교 근거인가요?


In [ ]:
COMPARISON_ARTIFACT_PATH = "artifacts/experiments/chapter_02/validation_degradation_comparison.json"
MODEL_TEST_EVAL_PATH = "artifacts/experiments/chapter_02/model_test_eval.json"
REPORT_ARTIFACT_PATH = "artifacts/reports/chapter_02_model_quality_comparison.md"

comparison_json, comparison_source = load_json_artifact(COMPARISON_ARTIFACT_PATH)
model_test_json, model_test_source = load_json_artifact(MODEL_TEST_EVAL_PATH)
report_text, report_source = load_text_artifact(REPORT_ARTIFACT_PATH)

model_info = comparison_json.get("model", {})
model_condition = pd.DataFrame(
    [
        {
            "model_name": model_info.get("model_name"),
            "model_version": model_info.get("model_version"),
            "model_path": model_info.get("model_path"),
            "target_column": model_info.get("target_column"),
            "operating_threshold": model_info.get("operating_threshold"),
            "feature_count": len(model_info.get("feature_columns", [])),
            "artifact_source": comparison_source,
        }
    ]
)

score_distribution_rows: list[dict[str, object]] = []
for dataset_name, stats in comparison_json.get("score_distribution", {}).items():
    prediction_counts = stats.get("prediction_counts", {})
    score_distribution_rows.append(
        {
            "dataset": dataset_name,
            "score_mean": stats.get("mean"),
            "score_p10": stats.get("p10"),
            "score_p50": stats.get("p50"),
            "score_p90": stats.get("p90"),
            "high_risk_prediction": prediction_counts.get(POSITIVE_LABEL),
            "low_risk_prediction": prediction_counts.get(NEGATIVE_LABEL),
        }
    )
score_distribution_artifact = pd.DataFrame(score_distribution_rows)
score_distribution_validation = score_distribution_artifact.loc[
    score_distribution_artifact["dataset"].isin(["valid_baseline", "valid_degraded"])
].copy()
score_distribution_delta = (
    score_distribution_validation.set_index("dataset")
    .loc["valid_degraded", ["score_mean", "score_p10", "score_p50", "score_p90", "high_risk_prediction", "low_risk_prediction"]]
    - score_distribution_validation.set_index("dataset")
    .loc["valid_baseline", ["score_mean", "score_p10", "score_p50", "score_p90", "high_risk_prediction", "low_risk_prediction"]]
).to_frame().T.round(4).assign(comparison="valid_degraded_minus_baseline")
score_distribution_delta = score_distribution_delta.loc[:, ["comparison", "score_mean", "score_p10", "score_p50", "score_p90", "high_risk_prediction", "low_risk_prediction"]]

artifact_provenance = pd.DataFrame(
    [
        {"artifact": "validation_degradation_comparison", "path": comparison_source, "qa_use": "model condition, score distribution, metric delta 확인"},
        {"artifact": "model_test_eval", "path": model_test_source, "qa_use": "선택된 test 평가 조건 확인"},
        {"artifact": "comparison_report", "path": report_source, "qa_use": "릴리스 회의 보고 문장 인용 전 원문 확인"},
    ]
)


### 출력 확인: `artifact_provenance`

**읽는 순서**  
먼저 `source` 또는 `path`를 보고, 그 다음 `execution_scope`나 데이터셋 이름을 확인합니다. 이 출력은 숫자를 해석하기 전에 “어떤 근거를 보고 있는지”를 고정하는 단계입니다.

**해석 기준**  
경로가 prepared artifact이면 준비된 근거를 읽은 것이고, 로컬 파일이면 현재 환경에서 읽은 근거입니다. 실행 범위가 sample이면 뒤의 metric 숫자를 전체 데이터 결과처럼 쓰면 안 됩니다.

**보고서 문장 예시**  
“이 결과는 `{path}` 경로의 prepared/local artifact를 기준으로 확인했으며, 실행 범위는 출력의 scope를 따릅니다.”

**주의할 점**  
값이 좋아 보이거나 나빠 보여도, 출처와 범위를 먼저 적지 않으면 보고서에서 재현 가능한 근거가 되기 어렵습니다.


In [ ]:
display(artifact_provenance)


### 출력 확인: `model_condition`

**읽는 순서**  
model artifact, threshold, feature 기준, artifact path를 확인합니다.

**해석 기준**  
같은 metric이라도 모델 버전이나 threshold가 다르면 비교할 수 없습니다. model condition은 비교의 전제입니다.

**보고서 문장 예시**  
“모델과 threshold 조건을 확인해 metric 비교의 기준을 고정했습니다.”

**주의할 점**  
조건이 다른 결과를 나란히 놓고 데이터 품질 변화라고 해석하면 안 됩니다.


In [ ]:
display(model_condition)


### 출력 확인: `score_distribution_validation`

**읽는 순서**  
score의 평균, 분위수, 최소/최대를 봅니다. 비교 표라면 baseline과 degraded의 변화 방향을 확인합니다.

**해석 기준**  
score는 threshold 적용 전 모델 출력입니다. score가 이동하면 입력 분포, 모델 버전, preprocessing 변화 후보를 생각할 수 있습니다.

**보고서 문장 예시**  
“score 분포에서 모델 출력이 어느 방향으로 움직였는지 확인했고, prediction 변화와 함께 해석했습니다.”

**주의할 점**  
score를 실제 확률이라고 단정하지 않습니다. 이 실습에서는 분포 변화와 threshold 영향에 집중합니다.


In [ ]:
display(score_distribution_validation)


### 출력 확인: `score_distribution_delta`

**읽는 순서**  
score의 평균, 분위수, 최소/최대를 봅니다. 비교 표라면 baseline과 degraded의 변화 방향을 확인합니다.

**해석 기준**  
score는 threshold 적용 전 모델 출력입니다. score가 이동하면 입력 분포, 모델 버전, preprocessing 변화 후보를 생각할 수 있습니다.

**보고서 문장 예시**  
“score 분포에서 모델 출력이 어느 방향으로 움직였는지 확인했고, prediction 변화와 함께 해석했습니다.”

**주의할 점**  
score를 실제 확률이라고 단정하지 않습니다. 이 실습에서는 분포 변화와 threshold 영향에 집중합니다.


In [ ]:
display(score_distribution_delta)


이 출력에서 확인할 핵심은 score와 prediction 분포가 metric의 중간 증거라는 점입니다. 품질 저하 데이터에서 score 평균이나 `high_risk` prediction count가 움직이면, feature 품질 신호가 모델 출력으로 이어졌는지 확인할 근거가 생깁니다.

다만 score 평균이 크게 움직이지 않아도 FP/FN은 바뀔 수 있습니다. 임계값 근처의 sample 이동, label basis 후보, 특정 feature 오류가 함께 작동할 수 있기 때문입니다.

### 4-2. metric delta와 FP/FN 변화 확인

이 셀의 판단은 데이터 품질 신호가 실제 오류 유형 변화와 함께 관측되는지 확인하는 것입니다. Accuracy 하나가 아니라 Precision, Recall, PR-AUC, FP, FN을 같이 봐야 운영 리스크를 설명할 수 있습니다.

prepared artifact의 metric은 같은 baseline model과 threshold `0.50`에서 계산된 값입니다. 따라서 이 비교에서는 모델 버전과 threshold 변경을 원인 후보에서 일단 제외하고, 데이터 조건과 label basis 후보를 우선 봅니다.

이 결과는 release 승인/보류의 최종 결론이 아닙니다. 3장에서 serving API가 같은 `model_version`, feature order, threshold, response fields를 사용하는지 확인해야 operational evidence로 넘어갈 수 있습니다.


### 따라하기 안내: metric 변화 표

**이 셀에서 할 일**  
baseline과 degraded metric을 나란히 봅니다.

**실행 후 볼 것**  
precision, recall, FP, FN 변화 방향을 봅니다.

**기록 포인트**  
어떤 오류 유형이 늘었는지 적습니다.

**생각해 볼 질문**  
어떤 metric이 같이 나빠졌나요?


In [ ]:
metric_rows: list[dict[str, object]] = []
for dataset_name, dataset_info in comparison_json.get("datasets", {}).items():
    metrics = dataset_info.get("metrics", {})
    confusion = dataset_info.get("confusion_matrix", {})
    metric_rows.append(
        {
            "dataset": dataset_name,
            "threshold": dataset_info.get("threshold"),
            "row_count": dataset_info.get("row_count"),
            "accuracy": metrics.get("accuracy"),
            "precision": metrics.get("precision"),
            "recall": metrics.get("recall"),
            "f1_score": metrics.get("f1_score"),
            "auroc": metrics.get("auroc"),
            "pr_auc": metrics.get("pr_auc"),
            "TP": confusion.get("true_positive"),
            "FP": confusion.get("false_positive"),
            "FN": confusion.get("false_negative"),
            "TN": confusion.get("true_negative"),
        }
    )
metric_artifact_table = pd.DataFrame(metric_rows)
validation_metric_table = metric_artifact_table.loc[
    metric_artifact_table["dataset"].isin(["valid_baseline", "valid_degraded"])
].copy()

artifact_delta = comparison_json.get("deltas", {})
metric_delta = pd.DataFrame(
    [
        {
            "comparison": "valid_degraded_minus_baseline",
            "accuracy": artifact_delta.get("accuracy"),
            "precision": artifact_delta.get("precision"),
            "recall": artifact_delta.get("recall"),
            "f1_score": artifact_delta.get("f1_score"),
            "auroc": artifact_delta.get("auroc"),
            "pr_auc": artifact_delta.get("pr_auc"),
            "FP": artifact_delta.get("false_positive"),
            "FN": artifact_delta.get("false_negative"),
        }
    ]
)

error_type_interpretation = pd.DataFrame(
    [
        {"metric_or_error": "Precision", "observed_delta": artifact_delta.get("precision"), "qa_meaning": "관심 class 예측 안에 실제 low_risk가 섞이는 정도를 봅니다."},
        {"metric_or_error": "Recall", "observed_delta": artifact_delta.get("recall"), "qa_meaning": "실제 high_risk sample을 놓치는 정도를 봅니다."},
        {"metric_or_error": "FP", "observed_delta": artifact_delta.get("false_positive"), "qa_meaning": "오탐 증가로 운영 확인 부담이 늘 수 있습니다."},
        {"metric_or_error": "FN", "observed_delta": artifact_delta.get("false_negative"), "qa_meaning": "미탐 증가로 위험 알림 누락 후보가 남습니다."},
        {"metric_or_error": "PR-AUC", "observed_delta": artifact_delta.get("pr_auc"), "qa_meaning": "threshold 하나가 아니라 관심 class 구분력 약화 후보를 봅니다."},
    ]
)


### 출력 확인: `validation_metric_table`

**읽는 순서**  
accuracy, precision, recall, F1, AUROC/PR-AUC를 봅니다. 하나만 고르지 말고 FP/FN과 연결합니다.

**해석 기준**  
precision은 FP 영향을 크게 받고, recall은 FN 영향을 크게 받습니다. 이 실습에서는 accuracy보다 어떤 오류가 늘었는지가 중요합니다.

**보고서 문장 예시**  
“metric 표에서 precision/recall과 FP/FN을 함께 확인해 모델 품질 판단 근거로 사용했습니다.”

**주의할 점**  
metric 값만 복사하지 않습니다. 데이터셋, threshold, positive label 조건을 함께 적어야 합니다.


In [ ]:
display(validation_metric_table)


### 출력 확인: `metric_delta`

**읽는 순서**  
baseline 대비 증가/감소 방향을 봅니다. 양수와 음수가 각각 좋은 변화인지 나쁜 변화인지 항목별로 해석합니다.

**해석 기준**  
delta는 원인을 확정하지 않고 변화가 발생한 위치를 알려 줍니다. metric delta는 데이터 품질 delta, score delta와 함께 봐야 합니다.

**보고서 문장 예시**  
“baseline 대비 변화량에서 어떤 품질 신호와 metric이 함께 움직였는지 확인했습니다.”

**주의할 점**  
모든 delta의 부호가 같은 의미는 아닙니다. 예를 들어 recall delta 감소와 FP delta 증가는 서로 다른 오류 변화를 뜻합니다.


In [ ]:
display(metric_delta)


### 출력 확인: `error_type_interpretation`

**읽는 순서**  
오류 유형별 설명 문장을 읽고 FP/FN 변화가 어떤 운영 의미를 갖는지 확인합니다.

**해석 기준**  
이 표는 metric 숫자를 사람이 이해할 수 있는 오류 설명으로 바꿔 줍니다. 최종 보고서에 가장 가까운 해석 출력입니다.

**보고서 문장 예시**  
“오류 유형 해석을 통해 어떤 실패가 늘었고 어떤 후속 확인이 필요한지 정리했습니다.”

**주의할 점**  
이 문장만 떼어 쓰지 말고, 앞의 metric delta와 데이터 품질 신호를 근거로 함께 인용합니다.


In [ ]:
display(error_type_interpretation)


이 출력에서 확인할 핵심은 품질 저하 validation에서 FP와 FN이 모두 증가한다는 점입니다. Precision과 Recall이 함께 낮아지면 “오탐만 줄이면 된다” 또는 “미탐만 보면 된다”처럼 단순화하면 안 됩니다.

같은 model_version과 threshold에서 metric이 흔들렸으므로, 데이터 조건 변화와 label basis 후보를 우선 확인합니다. 하지만 이 결과만으로 실제 serving release를 보류하거나 승인할 수는 없습니다. API 계약과 운영 로그 확인이 다음 gate입니다.

## 5. 원인 후보 연결과 QA 판단

### 5-1. GE failure, feature delta, metric delta를 하나의 후보표로 연결

이 셀의 판단은 관측값을 원인 후보와 owner가 있는 QA action으로 바꾸는 것입니다. 데이터 품질 신호와 metric 변화가 같은 방향으로 보인다고 해서 원인이 확정되는 것은 아닙니다. 대신 후보를 강화하거나 남겨두고, 다음 확인을 지정해야 합니다.

후보표는 report-ready evidence입니다. reviewer는 이 표를 보고 어떤 근거가 어떤 결론을 지탱하는지, 어떤 owner가 무엇을 확인해야 하는지 추적할 수 있어야 합니다.

여기서는 feature 품질, label basis, threshold trade-off, model_version/API contract를 분리합니다. 분리하지 않으면 모델 owner에게 모든 문제를 몰거나, 반대로 데이터 문제로만 단정하는 오류가 생깁니다.


### 따라하기 안내: 품질 신호 연결

**이 셀에서 할 일**  
GE 실패와 metric 변화를 함께 봅니다.

**실행 후 볼 것**  
같은 feature나 오류 유형으로 연결되는지 확인합니다.

**기록 포인트**  
원인 후보와 caveat를 적습니다.

**생각해 볼 질문**  
이 연결은 확정 원인인가요, 다음 확인 후보인가요?


In [ ]:
failed_ge_columns = ge_failure_table["column"].tolist() if not ge_failure_table.empty else []
feature_quality_status = quality_gate.loc[0, "status"]
metric_delta_row = metric_delta.iloc[0].to_dict()
score_delta_row = score_distribution_delta.iloc[0].to_dict()

cause_candidate_table = pd.DataFrame(
    [
        {
            "candidate": "feature_quality_shift",
            "evidence": f"GE failures={failed_ge_columns}; missing_row_delta={quality_delta['rows_with_any_feature_missing'].iloc[0]}; oxygen_saturation_max_delta={quality_delta['oxygen_saturation_max'].iloc[0]}",
            "candidate_status": "strengthened" if feature_quality_status == "review" else "weak",
            "owner": "Data Engineering",
            "next_action": "heart_rate 결측과 oxygen_saturation 범위 오류의 수집/가공 경로를 확인합니다.",
            "report_use": "입력 feature 품질 이슈가 score와 metric 해석 제한 사항입니다.",
        },
        {
            "candidate": "prediction_distribution_shift",
            "evidence": f"high_risk_prediction_delta={score_delta_row.get('high_risk_prediction')}, score_mean_delta={score_delta_row.get('score_mean')}",
            "candidate_status": "strengthened" if score_delta_row.get("high_risk_prediction", 0) != 0 else "open",
            "owner": "ML/QA",
            "next_action": "임계값 근처 sample과 score 분위수 변화를 확인합니다.",
            "report_use": "feature 품질 변화가 prediction 분포로 이어졌는지 확인합니다.",
        },
        {
            "candidate": "metric_error_shift",
            "evidence": f"precision_delta={metric_delta_row.get('precision')}, recall_delta={metric_delta_row.get('recall')}, FP_delta={metric_delta_row.get('FP')}, FN_delta={metric_delta_row.get('FN')}",
            "candidate_status": "strengthened" if metric_delta_row.get("FP", 0) or metric_delta_row.get("FN", 0) else "weak",
            "owner": "QA",
            "next_action": "FP/FN sample을 데이터 품질 신호와 함께 검토합니다.",
            "report_use": "Accuracy 단독이 아니라 오류 유형 변화로 리스크를 설명합니다.",
        },
        {
            "candidate": "label_basis_review",
            "evidence": "allowed label checks pass, but label flip candidate cannot be ruled out by allowed-value validation",
            "candidate_status": "open",
            "owner": "Data Owner/QA",
            "next_action": "라벨 생성 기준과 일부 반전 후보를 별도 샘플로 검토합니다.",
            "report_use": "허용 label 통과와 정답 기준 신뢰를 분리해 씁니다.",
        },
        {
            "candidate": "serving_contract_mismatch",
            "evidence": "not_checked_in_chapter_02",
            "candidate_status": "open_for_chapter_03",
            "owner": "Serving/Platform",
            "next_action": "API response에서 model_version, threshold, feature contract를 확인합니다.",
            "report_use": "2장 artifact만으로 release 승인/보류를 확정하지 않습니다.",
        },
    ]
)

display(cause_candidate_table)


이 출력에서 확인할 핵심은 후보 상태가 단계별로 바뀐다는 점입니다. `feature_quality_shift`와 `metric_error_shift`는 이번 장에서 강화되지만, `label_basis_review`와 `serving_contract_mismatch`는 아직 열린 후보로 남습니다.

보고서에서는 이 후보표를 그대로 결론으로 쓰지 않습니다. 관측값, 해석 제한, owner, next action을 포함한 문장으로 바꿔야 release 회의에서 방어 가능한 QA 의견이 됩니다.

### 5-2. 2장 데이터-지표 연결 evidence packet 만들기

마지막 판단은 이 notebook의 결과를 다음 장으로 넘길 수 있는 evidence packet으로 조립하는 것입니다. 이 packet은 모델 수정 지시서가 아니라, 데이터 품질 신호와 metric 변화가 같은 사건에서 함께 관측되었다는 QA handoff입니다.

최종 판단은 조건부입니다. 입력 품질 신호와 FP/FN 변화가 함께 보이므로 모델 자체 결함 단정은 보류하고, 데이터 수집/가공 경로와 serving contract를 다음 gate에서 확인합니다.

이 출력은 보고서 초안으로 사용할 수 있어야 합니다. artifact 경로와 실행 범위가 포함되어야 reviewer가 나중에 같은 근거를 확인할 수 있습니다.


### 따라하기 안내: 보고 문장

**이 셀에서 할 일**  
데이터 품질과 metric 변화를 한 문장으로 묶습니다.

**실행 후 볼 것**  
조건, 관측, 다음 확인이 들어 있는지 봅니다.

**기록 포인트**  
release 판단으로 넘길 문장을 적습니다.

**생각해 볼 질문**  
이 문장에 어떤 제한 사항을 붙여야 할까요?


In [ ]:
report_sentence = (
    "같은 기준선 모델과 threshold 0.50에서 validation baseline과 degraded 데이터를 비교했습니다. "
    f"GE artifact에서는 {', '.join(failed_ge_columns)} 관련 feature 품질 실패가 확인되었고, "
    f"notebook 계산에서는 feature missing row delta={quality_delta['rows_with_any_feature_missing'].iloc[0]}, "
    f"oxygen_saturation max delta={quality_delta['oxygen_saturation_max'].iloc[0]}가 관측되었습니다. "
    f"prepared model artifact에서는 precision delta={metric_delta_row.get('precision')}, "
    f"recall delta={metric_delta_row.get('recall')}, FP delta={metric_delta_row.get('FP')}, "
    f"FN delta={metric_delta_row.get('FN')}가 확인되었습니다. "
    "따라서 모델 자체 결함으로 단정하지 않고, feature 품질 변화와 label basis 후보를 확인한 뒤 "
    "3장에서 serving API의 model_version, threshold, response field contract를 확인합니다."
)

chapter_02_data_metric_packet = pd.DataFrame(
    [
        {
            "evidence": "ge_validation_failure",
            "observed": f"failure_count={ge_result_json.get('failure_count')}, failed_columns={failed_ge_columns}",
            "qa_judgment": "feature 품질 실패가 score와 metric 변화의 원인 후보입니다.",
            "owner": "Data Engineering",
            "next_action": "결측과 범위 오류의 수집/가공 경로를 확인합니다.",
        },
        {
            "evidence": "data_quality_delta",
            "observed": f"missing_row_delta={quality_delta['rows_with_any_feature_missing'].iloc[0]}, oxygen_saturation_max_delta={quality_delta['oxygen_saturation_max'].iloc[0]}",
            "qa_judgment": "artifact의 GE failure가 notebook dataframe 계산에서도 재현됩니다.",
            "owner": "QA/Data Engineering",
            "next_action": "품질 신호가 FP/FN sample에 집중되는지 확인합니다.",
        },
        {
            "evidence": "score_prediction_distribution",
            "observed": f"high_risk_prediction_delta={score_delta_row.get('high_risk_prediction')}, score_mean_delta={score_delta_row.get('score_mean')}",
            "qa_judgment": "feature 변화가 prediction 분포로 이어졌는지 확인합니다.",
            "owner": "ML/QA",
            "next_action": "threshold 근처 sample과 score 분위수를 검토합니다.",
        },
        {
            "evidence": "metric_delta",
            "observed": f"precision_delta={metric_delta_row.get('precision')}, recall_delta={metric_delta_row.get('recall')}, FP_delta={metric_delta_row.get('FP')}, FN_delta={metric_delta_row.get('FN')}",
            "qa_judgment": "오탐과 미탐 변화가 함께 관측되어 Accuracy 단독 판단은 보류합니다.",
            "owner": "QA",
            "next_action": "3장에서 API 계약과 운영 로그 evidence를 확인합니다.",
        },
        {
            "evidence": "artifact_provenance",
            "observed": f"GE={ge_result_source}; metric={comparison_source}",
            "qa_judgment": "full runtime 재생성이 아니라 prepared artifact 확인값임을 보고서에 구분합니다.",
            "owner": "QA",
            "next_action": "일반 GE/MLflow notebook 또는 로컬 Demo에서 재생성 경로를 별도 확인합니다.",
        },
    ]
)

handoff_to_serving = pd.DataFrame(
    [
        {
            "from_chapter": "02_model_quality",
            "to_chapter": "03_serving",
            "decision": "proceed_to_serving_contract_check",
            "strengthened_candidates": "feature_quality_shift, metric_error_shift",
            "open_candidates": "label_basis_review, serving_contract_mismatch, threshold_tradeoff",
            "serving_questions": "API response가 model_version, threshold, score, prediction, request_id를 같은 계약으로 반환하는가?",
        }
    ]
)

print(report_sentence)


### 출력 확인: `chapter_02_data_metric_packet`

**읽는 순서**  
앞 단계 결과가 어떤 evidence 묶음으로 정리되었는지 봅니다. 다음 장으로 넘길 질문이나 제한 사항도 같이 확인합니다.

**해석 기준**  
packet과 handoff는 최종 보고서 초안에 가깝습니다. 숫자, 조건, 판단, 다음 확인이 함께 들어 있어야 합니다.

**보고서 문장 예시**  
“이 packet을 근거로 다음 장에서 serving 계약 또는 운영 관측 확인으로 넘어갑니다.”

**주의할 점**  
packet이 있다고 분석이 끝난 것은 아닙니다. handoff에 남은 질문을 다음 장에서 계속 확인합니다.


In [ ]:
display(chapter_02_data_metric_packet)


### 출력 확인: `handoff_to_serving`

**읽는 순서**  
앞 단계 결과가 어떤 evidence 묶음으로 정리되었는지 봅니다. 다음 장으로 넘길 질문이나 제한 사항도 같이 확인합니다.

**해석 기준**  
packet과 handoff는 최종 보고서 초안에 가깝습니다. 숫자, 조건, 판단, 다음 확인이 함께 들어 있어야 합니다.

**보고서 문장 예시**  
“이 packet을 근거로 다음 장에서 serving 계약 또는 운영 관측 확인으로 넘어갑니다.”

**주의할 점**  
packet이 있다고 분석이 끝난 것은 아닙니다. handoff에 남은 질문을 다음 장에서 계속 확인합니다.


In [ ]:
display(handoff_to_serving)


### 5-3. 실패 시 확인 포인트

실행이 실패하면 먼저 `artifact_provenance`와 `validation_provenance`를 확인합니다. CSV는 읽혔지만 artifact JSON이 missing이면 metric 비교는 prepared artifact 확인값이 아니므로 보고서에 쓸 수 없습니다.

GE 실패 수가 문서와 다르면 `artifacts/great_expectations/chapter_02_validation_result.json`이 최신인지 확인합니다. JupyterLite에서는 artifact를 재생성하지 않으므로, 준비된 파일이 오래되었을 수 있습니다.

metric delta가 예상과 다르면 이 notebook에서 직접 모델을 재학습했다고 말하지 않습니다. `validation_degradation_comparison.json`의 model condition, threshold, dataset row count를 확인하고, 일반 notebook 또는 로컬 Demo companion에서 full runtime 재생성 경로를 확인합니다.
